# 第 17 章 対数線形近似: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/17-log-linearization.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
alpha = 0.5          # 資本分配率 α
saving = 0.2         # 貯蓄率 s
depreciation = 0.05  # 減耗率 δ
initial_deviation = 0.5  # 定常状態からの、最初の資本の乖離（0.5 は 50%）
periods = 60         # 何期先まで計算するか

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 遷移係数と、線形近似の誤差

In [ ]:
import numpy as np

from lib.chapter16.solow_stochastic import SolowParameters
from lib.chapter17.loglinear import (
    capital_paths,
    linear_transition,
    log_linearize,
    solow_equations,
    steady_state_capital,
)

params = SolowParameters(
    alpha=alpha, saving=saving, depreciation=depreciation, hours=1.0
)
k_star = steady_state_capital(params)
transition = linear_transition(
    *log_linearize(solow_equations(params), np.array([1.0, k_star]))
)
print(f"遷移係数: {float(transition[1, 1]):.4f}")
print(f"1 − (1 − α)·δ: {1 - (1 - alpha) * depreciation:.4f}")

paths = capital_paths(params, initial_deviation=initial_deviation, periods=periods)
max_error = float((paths["exact"] - paths["linear"]).abs().max())
print(f"最大の誤差: {max_error:.4f} %（最初の乖離は {100 * initial_deviation:.0f} %）")

## 図: 厳密な経路と、線形近似の経路

In [ ]:
from lib.chapter17.plot import plot_exact_vs_linear

plot_exact_vs_linear(paths)

## 試してみよう

- `initial_deviation = 0.01` にすると、最大の誤差は 0.0006 % です。`0.1`
   なら 0.0609 % で、乖離が大きいほど、誤差が大きくなります
- `depreciation = 0.1` にすると、遷移係数は 0.95（= 1 − 0.5 × 0.1）で、
  最大の誤差は 1.4835 % です